# Extração IBGE — População e territorialidade para o projeto Conecta Saúde

Este notebook documenta a fonte **IBGE** para o projeto.

O objetivo é mostrar:

1. **quais dados serão extraídos**;
2. **quais APIs públicas serão usadas**;
3. **como realizar a extração com Python**;
4. **quais campos são úteis para o projeto**;
5. **quais perguntas de negócio podem ser respondidas com esses dados**.

O IBGE será usado principalmente para normalizar os indicadores do DATASUS. Sem população e território, um ranking absoluto de internações pode favorecer municípios grandes e esconder municípios pequenos com pressão proporcionalmente maior.


## 1. Papel do IBGE no projeto

O IBGE será a base de **contexto populacional e territorial**.

A base será usada para analisar:

- população municipal;
- UF, município, região e hierarquia territorial;
- código IBGE do município;
- população estimada ou censitária;
- indicadores proporcionais por habitante;
- mapas e filtros regionais.

Na arquitetura do projeto, o IBGE será cruzado com:

- **SIH/SUS**, para calcular internações por 10 mil habitantes;
- **CNES**, para calcular leitos por 10 mil habitantes;
- **SIA/SUS**, para calcular procedimentos ambulatoriais por 10 mil habitantes;
- **dados auxiliares do projeto**, como região de saúde, metas ou classificação territorial.


## 2. APIs públicas de conexão

Usaremos duas famílias de APIs do IBGE:

### API de Localidades

Base para obter municípios, UFs e regiões.

```text
https://servicodados.ibge.gov.br/api/v1/localidades/municipios
```

```text
https://servicodados.ibge.gov.br/api/v1/localidades/estados
```

### API de Agregados/SIDRA

Base para obter população por município. Para estimativas populacionais, a tabela mais usada é a **6579 — População residente estimada**.

Estrutura do endpoint:

```text
https://servicodados.ibge.gov.br/api/v3/agregados/6579/periodos/{ANO}/variaveis/9324?localidades=N6[all]
```

Exemplo para 2024:

```text
https://servicodados.ibge.gov.br/api/v3/agregados/6579/periodos/2024/variaveis/9324?localidades=N6[all]
```

> Observação: a disponibilidade do ano depende da publicação oficial do IBGE. Caso o ano escolhido ainda não esteja disponível, use o último ano publicado ou utilize a população do Censo 2022 como referência fixa para o MVP.


In [ ]:
# Instalação das dependências
# Execute esta célula no Colab/Jupyter caso os pacotes ainda não estejam instalados.

%pip install -q pandas pyarrow requests


## 3. Parâmetros iniciais


In [ ]:
from pathlib import Path
import pandas as pd
import requests

ANO_POPULACAO = 2024

BASE_URL_LOCALIDADES = "https://servicodados.ibge.gov.br/api/v1/localidades"
BASE_URL_AGREGADOS = "https://servicodados.ibge.gov.br/api/v3/agregados"

DIRETORIO_RAW = Path("dados/raw/ibge")
DIRETORIO_TRATADO = Path("dados/tratado/ibge")

DIRETORIO_RAW.mkdir(parents=True, exist_ok=True)
DIRETORIO_TRATADO.mkdir(parents=True, exist_ok=True)


## 4. Extração da hierarquia de municípios

A API de Localidades retorna a estrutura completa do município, incluindo microrregião, mesorregião, UF e região.


In [ ]:
def get_json(url: str, timeout: int = 60):
    resposta = requests.get(url, timeout=timeout)
    resposta.raise_for_status()
    return resposta.json()

url_municipios = f"{BASE_URL_LOCALIDADES}/municipios"
municipios_json = get_json(url_municipios)

len(municipios_json), municipios_json[0]


## 5. Normalização da estrutura territorial

A célula abaixo transforma o JSON hierárquico em uma tabela municipal plana, pronta para joins com DATASUS.


In [ ]:
def normalizar_municipios_ibge(municipios_json: list[dict]) -> pd.DataFrame:
    linhas = []
    for item in municipios_json:
        microrregiao = item.get("microrregiao") or {}
        mesorregiao = microrregiao.get("mesorregiao") or {}
        uf = mesorregiao.get("UF") or {}
        regiao = uf.get("regiao") or {}
        regiao_imediata = item.get("regiao-imediata") or {}
        regiao_intermediaria = regiao_imediata.get("regiao-intermediaria") or {}

        linhas.append({
            "cod_municipio_ibge": str(item.get("id")),
            "municipio": item.get("nome"),
            "cod_uf": uf.get("id"),
            "uf": uf.get("sigla"),
            "estado": uf.get("nome"),
            "cod_regiao": regiao.get("id"),
            "regiao": regiao.get("nome"),
            "cod_microrregiao": microrregiao.get("id"),
            "microrregiao": microrregiao.get("nome"),
            "cod_mesorregiao": mesorregiao.get("id"),
            "mesorregiao": mesorregiao.get("nome"),
            "cod_regiao_imediata": regiao_imediata.get("id"),
            "regiao_imediata": regiao_imediata.get("nome"),
            "cod_regiao_intermediaria": regiao_intermediaria.get("id"),
            "regiao_intermediaria": regiao_intermediaria.get("nome"),
        })
    return pd.DataFrame(linhas)

df_municipios = normalizar_municipios_ibge(municipios_json)
df_municipios.head()


## 6. Extração da população municipal pela API de Agregados

A função abaixo consulta a tabela 6579, variável 9324, para todos os municípios (`N6[all]`).


In [ ]:
def montar_url_populacao_estimada(ano: int) -> str:
    return f"{BASE_URL_AGREGADOS}/6579/periodos/{ano}/variaveis/9324?localidades=N6[all]"

url_pop = montar_url_populacao_estimada(ANO_POPULACAO)
print(url_pop)

pop_json = get_json(url_pop)
pop_json[0].keys()


## 7. Normalização da população municipal

A resposta da API de Agregados possui uma estrutura aninhada. A função abaixo transforma essa estrutura em uma tabela com código do município, ano e população.


In [ ]:
def normalizar_populacao_sidra(pop_json: list[dict], ano: int) -> pd.DataFrame:
    '''Normaliza a resposta da API de Agregados do IBGE para DataFrame.'''
    linhas = []

    for bloco in pop_json:
        variavel = bloco.get("variavel")
        nome_variavel = bloco.get("nome")
        unidade = bloco.get("unidade")

        for resultado in bloco.get("resultados", []):
            series = resultado.get("series", [])
            for serie in series:
                localidade = serie.get("localidade", {})
                serie_valores = serie.get("serie", {})
                valor = serie_valores.get(str(ano))

                linhas.append({
                    "cod_municipio_ibge": str(localidade.get("id")),
                    "municipio_api": localidade.get("nome"),
                    "ano_populacao": ano,
                    "variavel": variavel,
                    "nome_variavel": nome_variavel,
                    "unidade": unidade,
                    "populacao": pd.to_numeric(valor, errors="coerce"),
                })

    return pd.DataFrame(linhas)

df_populacao = normalizar_populacao_sidra(pop_json, ANO_POPULACAO)
df_populacao.head()


## 8. Base final IBGE para o projeto

A base final junta a hierarquia territorial com a população.


In [ ]:
df_ibge = df_municipios.merge(
    df_populacao[["cod_municipio_ibge", "ano_populacao", "populacao"]],
    on="cod_municipio_ibge",
    how="left"
)

# Código de 6 dígitos usado com frequência em bases DATASUS para município.
# Em vários sistemas, o DATASUS usa os seis primeiros dígitos do código IBGE de 7 dígitos.
df_ibge["cod_municipio_datasus_6"] = df_ibge["cod_municipio_ibge"].str[:6]

df_ibge.head()


## 9. Quais dados serão extraídos

| Campo | Uso no projeto |
|---|---|
| `cod_municipio_ibge` | Chave territorial oficial do município |
| `cod_municipio_datasus_6` | Chave auxiliar para cruzar com campos municipais de alguns sistemas DATASUS |
| `municipio` | Nome do município |
| `uf` | Unidade Federativa |
| `estado` | Nome do estado |
| `regiao` | Região do Brasil |
| `microrregiao` | Recorte regional intermediário |
| `mesorregiao` | Recorte regional histórico |
| `regiao_imediata` | Recorte territorial atual do IBGE |
| `regiao_intermediaria` | Recorte territorial atual do IBGE |
| `populacao` | População estimada/censitária usada para normalização |
| `ano_populacao` | Ano de referência da população |


In [ ]:
pd.DataFrame({
    "coluna": df_ibge.columns,
    "tipo_detectado": [str(df_ibge[col].dtype) for col in df_ibge.columns]
})


## 10. Salvamento da base tratada


In [ ]:
arquivo_saida = DIRETORIO_TRATADO / f"ibge_municipios_populacao_{ANO_POPULACAO}.parquet"
df_ibge.to_parquet(arquivo_saida, index=False)
print(f"Arquivo salvo em: {arquivo_saida}")


## 11. Perguntas que conseguimos responder com IBGE

Com a base extraída, conseguimos responder perguntas de contexto e normalização:

1. Qual é a população de cada município analisado?
2. Quais municípios pertencem a cada UF e região?
3. Como comparar municípios de tamanhos diferentes de forma justa?
4. Quais municípios têm mais internações por 10 mil habitantes?
5. Quais municípios têm menos leitos SUS por 10 mil habitantes?
6. Quais regiões possuem maior pressão proporcional, não apenas maior volume absoluto?
7. Quais regiões têm demanda ambulatorial por especialidade acima da média populacional?
8. Quais municípios têm maior relação entre demanda hospitalar e população residente?
9. Quais municípios pequenos apresentam sinais de pressão assistencial relevante?
10. Onde priorizar atenção considerando demanda, capacidade e população?


## 12. Exemplo 1 — População por UF


In [ ]:
pop_uf = (
    df_ibge
    .groupby(["uf", "estado"], dropna=False)
    .agg(
        qtd_municipios=("cod_municipio_ibge", "nunique"),
        populacao=("populacao", "sum")
    )
    .reset_index()
    .sort_values("populacao", ascending=False)
)

pop_uf.head(20)


## 13. Exemplo 2 — Normalização de indicadores DATASUS

A função abaixo calcula uma taxa por 10 mil habitantes. Ela será usada em bases agregadas do SIH/SUS e SIA/SUS.


In [ ]:
def calcular_taxa_por_10k(valor: pd.Series, populacao: pd.Series) -> pd.Series:
    return (valor / populacao) * 10_000

# Exemplo didático: simulação de internações agregadas por município
exemplo_internacoes = df_ibge[["cod_municipio_ibge", "municipio", "uf", "populacao"]].head(5).copy()
exemplo_internacoes["internacoes"] = [120, 300, 80, 45, 210]
exemplo_internacoes["internacoes_por_10k_hab"] = calcular_taxa_por_10k(
    exemplo_internacoes["internacoes"],
    exemplo_internacoes["populacao"]
)

exemplo_internacoes


## 14. Como o IBGE entra no índice de pressão assistencial

O IBGE entra como denominador populacional:

```text
Internações por 10 mil habitantes = (internações SIH/SUS / população IBGE) * 10.000
```

```text
Leitos SUS por 10 mil habitantes = (leitos SUS CNES / população IBGE) * 10.000
```

```text
Procedimentos especializados por 10 mil habitantes = (procedimentos SIA/SUS / população IBGE) * 10.000
```

Essas métricas tornam a comparação mais justa entre municípios grandes, médios e pequenos.
